# 03 — SAM2 Zero-Shot Evaluation on CholecSeg8k

Evaluate the **pretrained** SAM2 (`facebook/sam2-hiera-base-plus`) on surgical
anatomy *before any fine-tuning*. This point-prompted score is the baseline the
LoRA fine-tuning in Step 5 must beat.

**Protocol.** SAM2 is class-agnostic, so we measure a *promptable upper bound*:
for every target class present in a frame we sample one point inside the
ground-truth region, prompt SAM2, keep its highest-confidence mask, and compute
IoU against that region.

**Prerequisites (Google Colab).**
- Use a GPU runtime — Runtime -> Change runtime type -> T4 or A100.
- Clone this repo and open the notebook from inside it, then run the setup
  cell below to install the dependencies Colab does not preinstall.
- The SAM2 checkpoint (~330 MB) downloads from the HuggingFace Hub on first run.
- CholecSeg8k downloaded — `bash scripts/download_cholecseg8k.sh`.

In [ ]:
# --- Colab setup: install dependencies not preinstalled on Colab ---
# Safe to re-run; skip it if your environment already has these.
%pip install -q "transformers>=5.8,<6" peft datasets albumentations

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data.cholecseg8k import CLASS_NAMES, CholecSeg8kDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| 6-class schema:", CLASS_NAMES)

## 1. Load pretrained SAM2 (zero-shot — no fine-tuning)

In [ ]:
from transformers import Sam2Model, Sam2Processor

CHECKPOINT = "facebook/sam2-hiera-base-plus"
# First call downloads the checkpoint (~330 MB) from the HuggingFace Hub.
model = Sam2Model.from_pretrained(CHECKPOINT).to(DEVICE).eval()
processor = Sam2Processor.from_pretrained(CHECKPOINT)
n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {CHECKPOINT}: {n_params / 1e6:.0f}M params")

## 2. Load the CholecSeg8k validation split

With `transform=None` each item is the full-resolution frame plus its 6-class
mask (`cystic_artery` is absent in CholecSeg8k, so it cannot be scored here).

In [ ]:
val_ds = CholecSeg8kDataset(split="val", transform=None)
print("validation frames:", len(val_ds))

sample = val_ds[0]
print("image:", tuple(sample["image"].shape), "| mask:", tuple(sample["mask"].shape))

## 3. Point-prompted segmentation helper

`sam2_point_mask` prompts SAM2 with a single foreground point and returns the
mask SAM2 itself ranks highest (`iou_scores`).

In [ ]:
def sample_point(binary_mask, rng):
    """Pick one (x, y) pixel inside a binary region."""
    ys, xs = np.where(binary_mask)
    i = int(rng.integers(len(xs)))
    return [float(xs[i]), float(ys[i])]


@torch.no_grad()
def sam2_point_mask(image_uint8, point):
    """Prompt SAM2 with one point; return its highest-IoU predicted mask."""
    inputs = processor(
        images=image_uint8,
        input_points=[[[point]]],   # [batch][object][point][x, y]
        input_labels=[[[1]]],       # 1 = foreground
        return_tensors="pt",
    ).to(DEVICE)
    outputs = model(**inputs, multimask_output=True)
    masks = processor.post_process_masks(
        outputs.pred_masks, inputs["original_sizes"],
    )[0]                            # (num_objects, num_masks, H, W)
    best = int(outputs.iou_scores[0, 0].argmax())
    return masks[0, best].cpu().numpy().astype(bool)


def iou(pred, target):
    union = np.logical_or(pred, target).sum()
    return np.logical_and(pred, target).sum() / union if union else np.nan

## 4. Per-class zero-shot IoU

Raise `N_IMAGES` for a tighter estimate (slower). Regions smaller than 50
pixels are skipped — a point prompt on a sliver of pixels is ill-posed.

In [ ]:
N_IMAGES = 100
MIN_REGION_PX = 50
rng = np.random.default_rng(42)
per_class = {c: [] for c in range(1, len(CLASS_NAMES))}   # skip background (0)

for idx in range(min(N_IMAGES, len(val_ds))):
    item = val_ds[idx]
    image = (item["image"].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    mask6 = item["mask"].numpy()
    for c in per_class:
        region = mask6 == c
        if region.sum() < MIN_REGION_PX:
            continue
        pred = sam2_point_mask(image, sample_point(region, rng))
        per_class[c].append(iou(pred, region))

print(f"{'class':16s} {'zero-shot IoU':>14s} {'n':>6s}")
for c, vals in per_class.items():
    score = np.nanmean(vals) if vals else float("nan")
    print(f"{CLASS_NAMES[c]:16s} {score:14.4f} {len(vals):6d}")
all_vals = [v for vals in per_class.values() for v in vals]
print(f"{'mean (foreground)':16s} {np.nanmean(all_vals):14.4f} {len(all_vals):6d}")

## 5. Qualitative examples

In [ ]:
item = val_ds[0]
image = (item["image"].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
mask6 = item["mask"].numpy()
present = [c for c in range(1, len(CLASS_NAMES))
           if (mask6 == c).sum() > MIN_REGION_PX]

fig, ax = plt.subplots(1, 2 + len(present), figsize=(4 * (2 + len(present)), 4))
ax[0].imshow(image)
ax[0].set_title("frame")
ax[1].imshow(mask6, vmin=0, vmax=5, cmap="tab10")
ax[1].set_title("ground truth (6-class)")
for k, c in enumerate(present):
    region = mask6 == c
    pred = sam2_point_mask(image, sample_point(region, np.random.default_rng(0)))
    ax[2 + k].imshow(image)
    ax[2 + k].imshow(pred, alpha=0.5, cmap="Reds")
    ax[2 + k].set_title(f"SAM2 zero-shot: {CLASS_NAMES[c]}")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## Notes

- This is the **zero-shot, point-prompted upper bound** for SAM2 on surgical
  anatomy — the baseline the LoRA fine-tuning in Step 5 must beat.
- SAM2 is class-agnostic: it segments *the object under the prompt point*, not
  a semantic class. `SAM2LoRASegmenter` (`src/models/sam2_lora.py`) repurposes
  the mask decoder into a prompt-free dense 6-class head.
- `cystic_artery` is absent from CholecSeg8k, so it cannot be evaluated here;
  it is learned from Endoscapes2023 (Step 6).
- Expect thin / rare structures (cystic duct) to score well below the liver —
  a single point prompt on a small region is ambiguous.